# MMVC Voice Conversion Interface

このノートブックは、訓練済みのMMVCモデルを使用してリアルタイム音声変換を行うためのインターフェースです。

## 1. 必要なライブラリのインポート

In [ ]:
import os
import json
import numpy as np
import torch
import soundfile as sf
import librosa
from scipy.io.wavfile import write
import IPython.display as ipd
from IPython.display import Audio, display
import matplotlib.pyplot as plt
import time

# MMVC modules
import sys
sys.path.append('..')

from models import SynthesizerTrn
from text import text_to_sequence
from text.symbols import symbols
import utils
import commons
import mel_processing

## 2. 設定の読み込み

In [ ]:
# 設定ファイルのパス
config_path = "../configs/baseconfig.json"

# 設定読み込み
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

print("設定を読み込みました:")
print(f"- モデル名: {config['model_name']}")
print(f"- サンプリング周波数: {config['sampling_rate']} Hz")
print(f"- フィルタ長: {config['filter_length']}")
print(f"- ホップ長: {config['hop_length']}")
print(f"- ウィンドウ長: {config['win_length']}")

## 3. 訓練済みモデルの読み込み

In [ ]:
# デバイス設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

# モデルパス（訓練後に生成されるチェックポイント）
# 注意: この部分は実際の訓練後に適切なパスに変更してください
model_path = "../logs/G_latest.pth"  # または特定のエポックのモデル

# モデルの初期化
net_g = SynthesizerTrn(
    len(symbols),
    config["filter_length"] // 2 + 1,
    config["segment_size"] // config["hop_length"],
    n_speakers=config.get("n_speakers", 0),
    **config["model"]
).to(device)

# チェックポイントの読み込み
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device)
    net_g.load_state_dict(checkpoint['model'])
    print(f"モデルを読み込みました: {model_path}")
    print(f"エポック: {checkpoint.get('epoch', 'N/A')}")
    print(f"ステップ: {checkpoint.get('iteration', 'N/A')}")
else:
    print(f"警告: モデルファイルが見つかりません: {model_path}")
    print("まず3_Train_MMVC.ipynbでモデルを訓練してください。")

# 推論モードに設定
net_g.eval()

## 4. 音声変換関数の定義

In [ ]:
def convert_voice(text, speaker_id=0, noise_scale=0.667, noise_scale_w=0.8, length_scale=1.0):
    """
    テキストから音声を生成
    
    Args:
        text (str): 変換するテキスト（日本語）
        speaker_id (int): スピーカーID（マルチスピーカーモデルの場合）
        noise_scale (float): ノイズスケール（音声の多様性）
        noise_scale_w (float): ウィンドウノイズスケール
        length_scale (float): 発話速度調整（1.0が標準、大きいほど遅い）
    
    Returns:
        numpy.ndarray: 生成された音声データ
    """
    # テキストを音素シーケンスに変換
    stn_tst = text_to_sequence(text, ["japanese_cleaners"])
    
    with torch.no_grad():
        x_tst = torch.LongTensor(stn_tst).unsqueeze(0).to(device)
        x_tst_lengths = torch.LongTensor([len(stn_tst)]).to(device)
        
        # スピーカーIDの設定（マルチスピーカーの場合）
        if config.get("n_speakers", 0) > 0:
            sid = torch.LongTensor([speaker_id]).to(device)
        else:
            sid = None
        
        # 音声生成
        audio = net_g.infer(
            x_tst, 
            x_tst_lengths, 
            sid=sid, 
            noise_scale=noise_scale, 
            noise_scale_w=noise_scale_w, 
            length_scale=length_scale
        )[0][0, 0].cpu().numpy()
    
    return audio

def convert_audio_to_audio(input_audio_path, target_speaker_id=0):
    """
    入力音声を指定されたスピーカーの声に変換
    
    Args:
        input_audio_path (str): 入力音声ファイルのパス
        target_speaker_id (int): 変換先のスピーカーID
    
    Returns:
        numpy.ndarray: 変換された音声データ
    """
    # 音声ファイルの読み込み
    audio, sr = librosa.load(input_audio_path, sr=config["sampling_rate"])
    
    # メル変換
    spec = mel_processing.spectrogram_torch(
        torch.FloatTensor(audio).unsqueeze(0),
        config["filter_length"],
        config["sampling_rate"],
        config["hop_length"],
        config["win_length"],
        center=False
    ).to(device)
    
    with torch.no_grad():
        # スピーカーIDの設定
        if config.get("n_speakers", 0) > 0:
            sid = torch.LongTensor([target_speaker_id]).to(device)
        else:
            sid = None
        
        # 音声変換
        audio_converted = net_g.voice_conversion(spec, sid)[0][0, 0].cpu().numpy()
    
    return audio_converted

## 5. テキストから音声生成のテスト

In [ ]:
# テストテキスト
test_texts = [
    "こんにちは、私はMMVCです。",
    "今日はとても良い天気ですね。",
    "音声変換技術は素晴らしいです。"
]

# 各テキストで音声生成
for i, text in enumerate(test_texts):
    print(f"\n--- テスト {i+1}: {text} ---")
    
    start_time = time.time()
    audio = convert_voice(text, speaker_id=0)
    generation_time = time.time() - start_time
    
    print(f"生成時間: {generation_time:.2f}秒")
    print(f"音声長: {len(audio)/config['sampling_rate']:.2f}秒")
    
    # 音声ファイルとして保存
    output_path = f"../output_test_{i+1}.wav"
    sf.write(output_path, audio, config["sampling_rate"])
    print(f"保存先: {output_path}")
    
    # 音声再生
    display(Audio(audio, rate=config["sampling_rate"]))

## 6. インタラクティブ音声生成

In [ ]:
# インタラクティブな音声生成
print("インタラクティブ音声生成モード")
print("終了するには 'quit' を入力してください")

while True:
    text = input("\n変換したいテキストを入力してください: ")
    
    if text.lower() == 'quit':
        print("音声生成を終了します。")
        break
    
    if not text.strip():
        print("テキストを入力してください。")
        continue
    
    try:
        # スピーカーID入力（マルチスピーカーの場合）
        if config.get("n_speakers", 0) > 1:
            speaker_input = input(f"スピーカーID (0-{config['n_speakers']-1}): ")
            speaker_id = int(speaker_input) if speaker_input.strip() else 0
        else:
            speaker_id = 0
        
        # 音声生成
        print("音声を生成中...")
        audio = convert_voice(text, speaker_id=speaker_id)
        
        # 音声再生
        display(Audio(audio, rate=config["sampling_rate"]))
        
        # 保存するかどうか
        save = input("この音声を保存しますか？ (y/n): ")
        if save.lower() == 'y':
            filename = input("ファイル名 (.wav拡張子は自動付与): ")
            if not filename.endswith('.wav'):
                filename += '.wav'
            sf.write(filename, audio, config["sampling_rate"])
            print(f"保存しました: {filename}")
    
    except Exception as e:
        print(f"エラーが発生しました: {e}")

## 7. 音声パラメータ調整実験

In [ ]:
# パラメータ調整実験
test_text = "音声合成のパラメータを調整しています。"

# 異なるパラメータで生成
params_list = [
    {"noise_scale": 0.3, "length_scale": 0.8, "name": "高速・低多様性"},
    {"noise_scale": 0.667, "length_scale": 1.0, "name": "標準"},
    {"noise_scale": 1.0, "length_scale": 1.2, "name": "低速・高多様性"},
]

print(f"テストテキスト: {test_text}\n")

for i, params in enumerate(params_list):
    print(f"--- {params['name']} ---")
    print(f"noise_scale: {params['noise_scale']}, length_scale: {params['length_scale']}")
    
    audio = convert_voice(
        test_text,
        noise_scale=params['noise_scale'],
        length_scale=params['length_scale']
    )
    
    # 音声再生
    display(Audio(audio, rate=config["sampling_rate"]))
    
    # 保存
    output_path = f"../param_test_{i+1}_{params['name']}.wav"
    sf.write(output_path, audio, config["sampling_rate"])
    print(f"保存: {output_path}\n")

## 8. 音声品質分析

In [ ]:
def analyze_audio_quality(audio, title="音声分析"):
    """
    生成された音声の品質を分析
    """
    plt.figure(figsize=(15, 10))
    
    # 波形表示
    plt.subplot(3, 1, 1)
    time_axis = np.arange(len(audio)) / config["sampling_rate"]
    plt.plot(time_axis, audio)
    plt.title(f"{title} - 波形")
    plt.xlabel("時間 (秒)")
    plt.ylabel("振幅")
    plt.grid(True)
    
    # スペクトログラム
    plt.subplot(3, 1, 2)
    D = librosa.stft(audio, hop_length=config["hop_length"])
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    librosa.display.specshow(S_db, sr=config["sampling_rate"], 
                           hop_length=config["hop_length"], x_axis='time', y_axis='hz')
    plt.title(f"{title} - スペクトログラム")
    plt.colorbar(format='%+2.0f dB')
    
    # メルスペクトログラム
    plt.subplot(3, 1, 3)
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=config["sampling_rate"], 
                                            hop_length=config["hop_length"])
    mel_spec_db = librosa.amplitude_to_db(mel_spec, ref=np.max)
    librosa.display.specshow(mel_spec_db, sr=config["sampling_rate"], 
                           hop_length=config["hop_length"], x_axis='time', y_axis='mel')
    plt.title(f"{title} - メルスペクトログラム")
    plt.colorbar(format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()
    
    # 統計情報
    print(f"音声統計情報:")
    print(f"  長さ: {len(audio)/config['sampling_rate']:.2f}秒")
    print(f"  最大振幅: {np.max(np.abs(audio)):.4f}")
    print(f"  RMS: {np.sqrt(np.mean(audio**2)):.4f}")
    print(f"  動的範囲: {20*np.log10(np.max(np.abs(audio))/np.mean(np.abs(audio))):.2f} dB")

# 分析実行
test_audio = convert_voice("音声品質の分析を行っています。")
analyze_audio_quality(test_audio, "MMVC生成音声")

## 9. バッチ処理

In [ ]:
def batch_text_to_speech(text_list, output_dir="../batch_output", speaker_id=0):
    """
    複数のテキストを一括で音声に変換
    
    Args:
        text_list (list): 変換するテキストのリスト
        output_dir (str): 出力ディレクトリ
        speaker_id (int): スピーカーID
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"バッチ処理開始: {len(text_list)}件のテキスト")
    
    results = []
    total_time = 0
    
    for i, text in enumerate(text_list):
        print(f"\n処理中 {i+1}/{len(text_list)}: {text[:30]}...")
        
        start_time = time.time()
        audio = convert_voice(text, speaker_id=speaker_id)
        generation_time = time.time() - start_time
        total_time += generation_time
        
        # ファイル保存
        filename = f"batch_{i+1:03d}.wav"
        output_path = os.path.join(output_dir, filename)
        sf.write(output_path, audio, config["sampling_rate"])
        
        results.append({
            "text": text,
            "filename": filename,
            "duration": len(audio) / config["sampling_rate"],
            "generation_time": generation_time
        })
        
        print(f"  完了: {filename} ({generation_time:.2f}秒)")
    
    print(f"\nバッチ処理完了!")
    print(f"総処理時間: {total_time:.2f}秒")
    print(f"平均処理時間: {total_time/len(text_list):.2f}秒/テキスト")
    print(f"出力ディレクトリ: {output_dir}")
    
    return results

# バッチ処理のテスト
batch_texts = [
    "バッチ処理のテストです。",
    "複数のテキストを一度に処理します。",
    "音声変換システムの性能を確認中です。",
    "MMVCは高品質な音声を生成します。",
    "バッチ処理が完了しました。"
]

batch_results = batch_text_to_speech(batch_texts)

## 10. システム情報とパフォーマンス

In [ ]:
def system_info():
    """
    システム情報とパフォーマンス統計を表示
    """
    print("=== MMVC システム情報 ===")
    print(f"PyTorch版本: {torch.__version__}")
    print(f"デバイス: {device}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name()}")
        print(f"CUDA版本: {torch.version.cuda}")
        print(f"GPU メモリ: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    print(f"\n=== モデル情報 ===")
    print(f"モデル名: {config['model_name']}")
    print(f"スピーカー数: {config.get('n_speakers', 1)}")
    print(f"サンプリング周波数: {config['sampling_rate']} Hz")
    
    # モデルパラメータ数
    total_params = sum(p.numel() for p in net_g.parameters())
    trainable_params = sum(p.numel() for p in net_g.parameters() if p.requires_grad)
    print(f"総パラメータ数: {total_params:,}")
    print(f"訓練可能パラメータ数: {trainable_params:,}")
    
    # パフォーマンステスト
    print(f"\n=== パフォーマンステスト ===")
    test_text = "パフォーマンステストを実行中です。"
    
    times = []
    for i in range(5):
        start = time.time()
        _ = convert_voice(test_text)
        times.append(time.time() - start)
    
    print(f"平均生成時間: {np.mean(times):.3f} ± {np.std(times):.3f} 秒")
    print(f"最短生成時間: {np.min(times):.3f} 秒")
    print(f"最長生成時間: {np.max(times):.3f} 秒")

system_info()

## 注意事項

1. このノートブックを使用する前に、`3_Train_MMVC.ipynb`でモデルの訓練を完了してください。
2. 訓練済みモデルのパス (`model_path`) を適切に設定してください。
3. マルチスピーカーモデルの場合、有効なスピーカーIDを指定してください。
4. 生成された音声の品質は訓練データの質と量に大きく依存します。
5. リアルタイム変換を行う場合は、十分なGPUメモリを確保してください。

## トラブルシューティング

- **メモリ不足エラー**: バッチサイズを小さくするか、より小さなモデルを使用してください。
- **音声品質が低い**: より多くのエポックで訓練するか、訓練データの品質を向上させてください。
- **生成が遅い**: GPUを使用していることを確認し、モデルサイズを調整してください。